# VTC Capability-Lift Harness  (qwen2.5-coder-1.5B, free Colab T4)

**The one question the VTC corpus has not yet answered:** does fine-tuning on the execution-verified
trajectories actually make a model *more capable* -- not just produce clean training data?

This notebook measures it honestly:
1. **Split** the corpus by MBPP `task_id`; eval ONLY on problems whose id is NOT in training (disjoint).
2. **QLoRA fine-tune** `Qwen/Qwen2.5-Coder-1.5B-Instruct` on the verified trajectories.
3. **Eval base vs trained** on held-out problems, *execution-verified* against hidden tests
   (LoRA-off vs LoRA-on on the SAME model, so the delta is the adapter, not prompt drift).

**Headline number = `newly_solved`:** held-out problems the TRAINED model passes that the BASE could not.

### Honest caveats (read before believing any number)
- MBPP is on GitHub and almost certainly in Qwen's pretraining; a 'solve' may be memorization, and the corpus
  was harvested FROM MBPP, so `newly_solved` upper-bounds novel capability -- it is not proof.
- A prior real 1.5B run here showed **zero** capability lift while verification never lied. Expect a small
  or null result and report it honestly -- a 1-2 problem swing on a small eval is within noise.
- `regressed` (base solved, trained broke) is always shown; net lift is never just the positive.
- Eval runs model-generated code in a subprocess -- fine on a throwaway Colab VM, not for untrusted output.


## 1. Install (unsloth brings peft/trl/bitsandbytes)


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes


## 2. Get the harness code + the corpus
`python/helm` comes from the repo clone. The corpus `vtc_mbpp_refined.jsonl` is training data (not
committed); the cell tries **Drive**, then a **URL**, then a manual **upload**. Set `DRIVE_CORPUS` or
`CORPUS_URL` to skip the upload prompt -- preferred for an unattended terminal-driven run (colab_run).


In [ ]:
import os, sys
!git clone --depth 1 https://github.com/issdandavis/SCBE-AETHERMOORE.git scbe || true
sys.path.insert(0, '/content/scbe')

# Corpus delivery in order of preference -- set ONE to run unattended:
DRIVE_CORPUS = '/content/drive/MyDrive/scbe/vtc_mbpp_refined.jsonl'  # option 2: a path on your Drive
CORPUS_URL   = ''  # option 3: a secret gist/release raw URL ('' to skip)
CORPUS_PATH  = '/content/vtc_mbpp_refined.jsonl'

if not os.path.exists(CORPUS_PATH):  # 2. Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        if os.path.exists(DRIVE_CORPUS):
            CORPUS_PATH = DRIVE_CORPUS
    except Exception as e:
        print('drive mount skipped:', e)
if not os.path.exists(CORPUS_PATH) and CORPUS_URL:  # 3. URL
    import urllib.request
    urllib.request.urlretrieve(CORPUS_URL, CORPUS_PATH)
if not os.path.exists(CORPUS_PATH):  # 1. fallback: manual upload
    try:
        from google.colab import files
        print('Upload vtc_mbpp_refined.jsonl (the verified-trajectory corpus):')
        up = files.upload(); CORPUS_PATH = '/content/' + next(iter(up))
    except Exception as e:
        raise SystemExit('No corpus via Drive/URL/upload: %s' % e)
os.environ['SCBE_VTC_CORPUS'] = CORPUS_PATH
print('corpus:', CORPUS_PATH)


## 3. Config (1.5B fits a free T4 comfortably; 7B OOMs)


In [ ]:
BASE_MODEL   = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'  # corpus models were 1.5b/3b; 1.5B = safe free-T4 pick
OUTPUT_DIR   = '/content/vtc-lift-run'
TRAIN_SFT    = '/content/train.sft.jsonl'
PUBLIC_K     = 1        # asserts the model sees; the rest are HIDDEN at eval time
EVAL_LIMIT   = 80       # held-out problems to eval (more = better signal, slower). None = all ~266
MAX_SEQ_LENGTH = 2048
LORA_RANK, LORA_ALPHA = 16, 32
LEARNING_RATE, NUM_EPOCHS = 2e-4, 2
BATCH_SIZE, GRAD_ACCUM = 2, 8
MAX_NEW_TOKENS = 512


## 4. Honest split: train ids vs disjoint held-out MBPP


In [ ]:
from python.helm import public_bench as pb
from python.helm.vtc_split import load_corpus, split_by_task_id, write_train_sft

records  = load_corpus(CORPUS_PATH)
problems = pb.pull_mbpp()  # full MBPP-sanitized (427); cached after first pull
split = split_by_task_id(records, problems, public_k=PUBLIC_K)
write_train_sft(split['train_records'], TRAIN_SFT)

eval_problems = split['eval_problems']
if EVAL_LIMIT:
    eval_problems = eval_problems[:EVAL_LIMIT]
assert not (split['train_ids'] & {p['task_id'] for p in eval_problems}), 'train/eval LEAK'
print('train records :', len(split['train_records']))
print('held-out eval :', len(eval_problems), '(of', len(split['eval_problems']), 'disjoint problems)')


## 5. Load base model + QLoRA adapter + the in-process generator


In [ ]:
from unsloth import FastLanguageModel
from python.helm.free_generator import strip_to_code

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42)

def make_hf_generator(model, tokenizer, public_k=1, max_new_tokens=512):
    """problem -> code, mirroring free_generator's prompt but calling the in-process checkpoint."""
    def gen(problem):
        public = '\n'.join(list(problem.get('test_list', []))[:public_k])
        prompt = ((problem.get('prompt') or problem.get('text') or '').strip()
                  + '\n\nWrite a complete Python solution. It must make this example pass:\n'
                  + public + '\nReturn ONLY the code.')
        ids = tokenizer.apply_chat_template([{'role':'user','content':prompt}],
                  tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        return strip_to_code(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    return gen


## 6. Train (QLoRA SFT on the verified trajectories)


In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

train_ds = load_dataset('json', data_files=TRAIN_SFT, split='train')
def convert(ex):
    out = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': out}
train_ds = train_ds.map(convert, remove_columns=train_ds.column_names)
print('train rows:', len(train_ds))

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, packing=True,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR, report_to='none', num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.05, learning_rate=LEARNING_RATE, optim='adamw_8bit',
        weight_decay=0.01, lr_scheduler_type='cosine', logging_steps=10, seed=42))
trainer.train()
model.save_pretrained(OUTPUT_DIR + '/lora'); tokenizer.save_pretrained(OUTPUT_DIR + '/lora')


## 7. The measurement: base (LoRA-off) vs trained (LoRA-on), execution-verified


In [ ]:
from python.helm.code_lift import solve_rate, lift_from_solve, render
FastLanguageModel.for_inference(model)
gen = make_hf_generator(model, tokenizer, public_k=PUBLIC_K, max_new_tokens=MAX_NEW_TOKENS)

print('Evaluating BASE (adapter disabled) over', len(eval_problems), 'held-out problems...')
with model.disable_adapter():
    base = solve_rate(eval_problems, gen, public_k=PUBLIC_K)
print('Evaluating TRAINED (adapter enabled)...')
trained = solve_rate(eval_problems, gen, public_k=PUBLIC_K)

report = lift_from_solve(base, trained)
print(); print(render(report)); print()
print('newly solved ids:', sorted(report['newly_solved']))
print('regressed ids   :', sorted(report['regressed']))


## 8. Recovery (the VTC thesis): does it solve AFTER failing?
Raw solve-lift may be ~0 (MBPP is likely in pretraining). The corpus is mostly repair traces, so the
specific bet is **recovery lift** -- of the problems it solves, the fraction it solved only after a failed
attempt. This runs the write->run->repair LOOP, base (LoRA-off) vs trained (LoRA-on).


In [ ]:
from python.helm.code_lift import measure_recovery_lift
from python.helm.free_generator import strip_to_code

def make_hf_ask(model, tokenizer, max_new_tokens=MAX_NEW_TOKENS):
    def ask(prompt):
        msgs = [{'role': 'user', 'content': prompt}]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                            return_tensors='pt').to(model.device)
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        return strip_to_code(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    return ask

ask = make_hf_ask(model, tokenizer)
with model.disable_adapter():
    base_ask = make_hf_ask(model, tokenizer)  # adapter disabled inside this block at call time
    rec = measure_recovery_lift(base_ask, ask, eval_problems, public_k=PUBLIC_K, rounds=3)
print(render(rec))  # render() prints RECOVERY LIFT when present


## 9. Export the trained model -- local Ollama + HuggingFace
Unsloth writes the merged weights, a quantized **GGUF** (one artifact that Ollama / LM Studio / Jan /
llama.cpp all consume), and can push straight to your HF account.


In [ ]:
# --- A) merged GGUF (q4_k_m) for local runtimes; also writes ./vtc_gguf/Modelfile for Ollama ---
model.save_pretrained_gguf('vtc_gguf', tokenizer, quantization_method='q4_k_m')

# --- B) push to your HuggingFace Hub (set HF_TOKEN in Colab Secrets) ---
# from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
# model.push_to_hub_gguf('issdandavis/vtc-qwen15-gguf', tokenizer,
#                        quantization_method='q4_k_m', token=HF_TOKEN)   # GGUF, pullable anywhere
# model.push_to_hub_merged('issdandavis/vtc-qwen15', tokenizer,
#                          save_method='merged_16bit', token=HF_TOKEN)   # full HF model (vLLM/TGI)
# model.push_to_hub('issdandavis/vtc-qwen15-lora', token=HF_TOKEN)        # the ~10MB adapter only

from google.colab import files  # download the GGUF + Modelfile to your machine
import glob
for f in glob.glob('vtc_gguf/*'):
    print(f)


### Import into LOCAL Ollama, then measure against the stock base (on your machine)
```bash
# OPTION 1 ($0 round-trip): if you ran push_to_hub_gguf above, pull it straight from HF into Ollama
#   -- no manual download, works on any machine:
ollama pull hf.co/issdandavis/vtc-qwen15-gguf:Q4_K_M

# OPTION 2 (local file): after downloading vtc_gguf/ (the .gguf + Modelfile):
ollama create vtc-qwen15 -f ./vtc_gguf/Modelfile

ollama pull qwen2.5-coder:1.5b      # the stock base for comparison

# solve-lift AND recovery-lift, base vs your trained model, on held-out MBPP
# (--trained = whichever name above: 'vtc-qwen15' for opt 2, or the full hf.co/... id for opt 1):
python -m python.helm.code_lift --base qwen2.5-coder:1.5b --trained vtc-qwen15 \
    --corpus training-data/sft/vtc_mbpp_refined.jsonl --recovery
```
### Where else it imports (the GGUF is one artifact; OpenAI-compatible servers need zero new code)
- **Ollama / LM Studio / Jan / GPT4All / llama.cpp** -- all consume the same `q4_k_m` GGUF directly.
- **vLLM** -- serve the merged model (or the LoRA adapter via `--enable-lora`); it exposes an
  OpenAI-compatible `/v1` endpoint, so `free_generator` + `code_lift` hit it with NO code change
  (just `SCBE_LLM_BASE=http://host:8000/v1 SCBE_LLM_MODEL=...`).
- **HF TGI** -- same OpenAI-compatible story for hosted inference.
- **transformers + PEFT** -- load base + the pushed adapter anywhere (`PeftModel.from_pretrained`).
- **scbe-verify MCP** -- point any of these endpoints at the verify tools so Grok/others get the same
  execution-verified loop.


## Notes (hold the honest line)
- `newly_solved > 0` is the only thing that shows capability lift -- and even then, see the memorization
  caveat above: id-disjoint != content-disjoint. Report the number, the `regressed` set, and `n` together.
- If `net_lift <= 0`, that is a real, publishable result: the corpus produces clean verified data but this
  fine-tune did not raise held-out capability. The verifier not lying is itself the contribution.
- To strengthen signal: raise `EVAL_LIMIT` to all ~266, train more epochs, or try the 3B base on a paid GPU.
- Determinism: `do_sample=False` so base and trained are compared on identical greedy decoding.
